# NW11 标定的深度域 dynamic-gain balanced seismic

本 notebook 改造 2026-04-29 的动态增益方法，并直接输出可导入地质软件的 balanced SEG-Y。

核心语义：

- 井上最小二乘尺度是 synthetic → observed 的 forward gain；
- balanced seismic 使用相对 reference forward gain 的逆增益；
- NW11 是唯一参与拟合的优选井；
- 采用 TVDSS 滑窗和稳健回归，不使用任意等分段；
- 逐道标准化只用于计算局部 relative RMS，输出仍保持原 SEG-Y 振幅单位；
- balance gain 受 NW11 标定范围与显式安全上限共同约束；
- 当前产物只做 dynamic gain，不把 coherence 混入振幅变换。

设置环境变量 DYNAMIC_GAIN_SMOKE_TRACES 为正整数时只处理前若干道并跳过 SEG-Y；未设置时处理完整体并输出 SEG-Y。


In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import cigsegy
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import least_squares

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
if not (repo_root / "src").exists():
    raise RuntimeError("Could not locate repository root.")

src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

from cup.petrel.load import import_seismic
from cup.seismic.survey import open_survey
from cup.seismic.volume_export import build_segy_textual_header

plt.rcParams["figure.dpi"] = 120
pd.set_option("display.max_columns", 120)

SEISMIC_FILE = repo_root / "data" / "raw" / "mero_84_coord_extend"
BATCH_DIR = repo_root / "scripts" / "output" / "wavelet_batch_synthetic_depth_20260719_172510"
BATCH_METRICS_FILE = BATCH_DIR / "wavelet_batch_metrics.csv"
OUTPUT_DIR = repo_root / "experiments" / "dynamic_gain_balancing" / "results" / "20260811_nw11_depth"
FIGURE_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

SELECTED_WELLS = ("NW11",)
CALIBRATION_WINDOW_M = 100.0
CALIBRATION_STEP_M = 25.0
APPLICATION_WINDOW_M = 100.0
MIN_WINDOW_SAMPLES = 12
MIN_LOCAL_CORR = 0.35
ROBUST_F_SCALE = 0.25
TRAINING_GAIN_QUANTILES = (2.0, 98.0)
ABSOLUTE_BALANCE_GAIN_CAP = (0.5, 3.0)
VOLUME_BATCH_TRACES = 2048
SMOKE_TRACE_LIMIT = int(os.environ.get("DYNAMIC_GAIN_SMOKE_TRACES", "0"))

SEGY_OPTIONS = {"iline": 5, "xline": 21, "istep": 1, "xstep": 4}
KEYLOCS = [SEGY_OPTIONS[key] for key in ("iline", "xline", "istep", "xstep")]

TRAINING_SAMPLES_FILE = OUTPUT_DIR / "nw11_dynamic_gain_training_windows.csv"
FIT_METRICS_FILE = OUTPUT_DIR / "dynamic_gain_fit_metrics.csv"
MODEL_FILE = OUTPUT_DIR / "dynamic_gain_balance_model.json"
BALANCED_SEGY_FILE = OUTPUT_DIR / "dynamic_gain_balanced_seismic_nw11_depth.segy"
TEMP_BALANCED_FILE = OUTPUT_DIR / ".dynamic_gain_balanced.float32.memmap"
SMOKE_NPZ_FILE = OUTPUT_DIR / "dynamic_gain_balanced_smoke.npz"
SUMMARY_FILE = OUTPUT_DIR / "run_summary.json"

for path in (SEISMIC_FILE, BATCH_METRICS_FILE):
    if not path.exists():
        raise FileNotFoundError(path)

print("Selected wells:", SELECTED_WELLS)
print("Input seismic:", SEISMIC_FILE)
print("Output directory:", OUTPUT_DIR)
print("Smoke trace limit:", SMOKE_TRACE_LIMIT)


In [ ]:
def resolve_artifact_path(value: str | Path) -> Path:
    path = Path(value)
    return path if path.is_absolute() else repo_root / path


def finite_corr(a: np.ndarray, b: np.ndarray) -> float:
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    valid = np.isfinite(a) & np.isfinite(b)
    if int(valid.sum()) < 3:
        return np.nan
    if np.std(a[valid]) <= 0.0 or np.std(b[valid]) <= 0.0:
        return np.nan
    return float(np.corrcoef(a[valid], b[valid])[0, 1])


def positive_ls_gain(observed: np.ndarray, synthetic: np.ndarray) -> float:
    observed = np.asarray(observed, dtype=np.float64)
    synthetic = np.asarray(synthetic, dtype=np.float64)
    valid = np.isfinite(observed) & np.isfinite(synthetic)
    if int(valid.sum()) < MIN_WINDOW_SAMPLES:
        return np.nan
    denominator = float(np.dot(synthetic[valid], synthetic[valid]))
    if not np.isfinite(denominator) or denominator <= 0.0:
        return np.nan
    gain = float(np.dot(observed[valid], synthetic[valid]) / denominator)
    return gain if np.isfinite(gain) and gain > 0.0 else np.nan


def resolve_odd_window_samples(window_axis_units: float, sample_step: float) -> int:
    count = max(3, int(round(float(window_axis_units) / float(sample_step))))
    return count if count % 2 == 1 else count + 1


def moving_sum_axis(values: np.ndarray, window_samples: int) -> np.ndarray:
    values = np.asarray(values, dtype=np.float32)
    left = window_samples // 2
    right = window_samples - 1 - left
    padded = np.pad(values, ((0, 0), (left, right)), mode="constant")
    cumsum = np.concatenate(
        [np.zeros((values.shape[0], 1), dtype=np.float64), np.cumsum(padded, axis=1, dtype=np.float64)],
        axis=1,
    )
    return cumsum[:, window_samples:] - cumsum[:, :-window_samples]


def moving_rms_axis(values: np.ndarray, window_samples: int) -> np.ndarray:
    values = np.asarray(values, dtype=np.float32)
    valid = np.isfinite(values)
    numerator = moving_sum_axis(np.where(valid, values * values, 0.0), window_samples)
    denominator = moving_sum_axis(valid.astype(np.float32), window_samples)
    result = np.full(values.shape, np.nan, dtype=np.float32)
    supported = denominator > 0.0
    result[supported] = np.sqrt(numerator[supported] / denominator[supported]).astype(np.float32)
    return result


def build_well_windows(metrics_row: pd.Series, window_m: float, step_m: float) -> pd.DataFrame:
    qc = pd.read_csv(resolve_artifact_path(metrics_row["synthetic_qc_path"]))
    depth_shift = pd.read_csv(resolve_artifact_path(metrics_row["depth_shift_curve_path"]))
    required_qc = {"twt_s", "seismic_norm", "synthetic_scaled"}
    required_depth = {"twt_s", "tvdss_m"}
    if missing := required_qc - set(qc.columns):
        raise ValueError(f"QC file is missing columns: {sorted(missing)}")
    if missing := required_depth - set(depth_shift.columns):
        raise ValueError(f"Depth-shift file is missing columns: {sorted(missing)}")

    twt = qc["twt_s"].to_numpy(dtype=np.float64)
    seismic = qc["seismic_norm"].to_numpy(dtype=np.float64)
    synthetic_raw = qc["synthetic_scaled"].to_numpy(dtype=np.float64) / float(metrics_row["scale"])
    depth_twt = depth_shift["twt_s"].to_numpy(dtype=np.float64)
    depth = depth_shift["tvdss_m"].to_numpy(dtype=np.float64)
    order = np.argsort(depth_twt)
    tvdss = np.interp(twt, depth_twt[order], depth[order])

    first = float(np.nanmin(tvdss)) + 0.5 * float(window_m)
    last = float(np.nanmax(tvdss)) - 0.5 * float(window_m)
    if last < first:
        raise ValueError(f"{metrics_row['well_name']} has insufficient depth support for {window_m:g} m windows.")

    rows: list[dict[str, float | int | str]] = []
    for center in np.arange(first, last + 0.25 * float(step_m), float(step_m)):
        mask = (
            (tvdss >= center - 0.5 * float(window_m))
            & (tvdss <= center + 0.5 * float(window_m))
            & np.isfinite(seismic)
            & np.isfinite(synthetic_raw)
        )
        if int(mask.sum()) < MIN_WINDOW_SAMPLES:
            continue
        correlation = finite_corr(seismic[mask], synthetic_raw[mask])
        forward_gain = positive_ls_gain(seismic[mask], synthetic_raw[mask])
        relative_rms = float(np.sqrt(np.mean(seismic[mask] ** 2)))
        if not (
            np.isfinite(correlation)
            and correlation >= MIN_LOCAL_CORR
            and np.isfinite(forward_gain)
            and forward_gain > 0.0
            and np.isfinite(relative_rms)
            and relative_rms > 0.0
        ):
            continue
        rows.append(
            {
                "well_name": str(metrics_row["well_name"]),
                "center_tvdss_m": float(center),
                "top_tvdss_m": float(center - 0.5 * window_m),
                "bottom_tvdss_m": float(center + 0.5 * window_m),
                "n_samples": int(mask.sum()),
                "local_corr": float(correlation),
                "relative_rms": relative_rms,
                "forward_gain": float(forward_gain),
            }
        )
    return pd.DataFrame(rows)


def fit_robust_log_gain(frame: pd.DataFrame) -> dict[str, float]:
    x = np.log(frame["relative_rms"].to_numpy(dtype=np.float64))
    y = np.log(frame["forward_gain"].to_numpy(dtype=np.float64))
    correlation = np.clip(frame["local_corr"].to_numpy(dtype=np.float64), 0.0, 1.0)
    count = frame["n_samples"].to_numpy(dtype=np.float64)
    weights = np.sqrt(correlation) * np.sqrt(count / np.mean(count))

    def residual(parameters: np.ndarray) -> np.ndarray:
        return weights * (parameters[0] + parameters[1] * x - y)

    initial = np.asarray([float(np.median(y)), 0.5], dtype=np.float64)
    solution = least_squares(residual, initial, loss="huber", f_scale=ROBUST_F_SCALE)
    intercept, slope = (float(solution.x[0]), float(solution.x[1]))
    prediction = intercept + slope * x
    return {
        "intercept": intercept,
        "slope": slope,
        "rmse_log_forward_gain": float(np.sqrt(np.mean((y - prediction) ** 2))),
        "mae_log_forward_gain": float(np.mean(np.abs(y - prediction))),
        "pearson_log_rms_log_gain": finite_corr(x, y),
    }


def apply_balance_to_batch(
    raw_values: np.ndarray,
    *,
    window_samples: int,
    slope: float,
    rms_floor: float,
    balance_gain_clip: tuple[float, float],
) -> tuple[np.ndarray, np.ndarray, int]:
    raw = np.asarray(raw_values, dtype=np.float32)
    finite = np.isfinite(raw)
    counts = finite.sum(axis=1, keepdims=True)
    safe_counts = np.maximum(counts, 1)
    means = np.where(finite, raw, 0.0).sum(axis=1, keepdims=True) / safe_counts
    centered = np.where(finite, raw - means, 0.0)
    scales = np.sqrt((centered * centered).sum(axis=1, keepdims=True) / safe_counts)
    live = np.isfinite(scales) & (scales > 0.0)
    normalized = np.zeros_like(raw, dtype=np.float32)
    np.divide(centered, scales, out=normalized, where=live)

    local_rms = moving_rms_axis(normalized, window_samples)
    local_rms = np.maximum(local_rms, float(rms_floor))
    balance_gain = np.power(local_rms, -float(slope), dtype=np.float32)
    balance_gain = np.clip(balance_gain, float(balance_gain_clip[0]), float(balance_gain_clip[1]))
    balance_gain = np.where(live, balance_gain, 1.0).astype(np.float32)

    balanced = means.astype(np.float32) + centered * balance_gain
    balanced = np.where(finite, balanced, 0.0).astype(np.float32)
    return balanced, balance_gain, int(np.count_nonzero(~live))


def plot_section_pair(
    raw_section: np.ndarray,
    balanced_section: np.ndarray,
    *,
    horizontal_axis: np.ndarray,
    vertical_axis: np.ndarray,
    title: str,
    output_path: Path,
) -> None:
    raw = np.asarray(raw_section, dtype=np.float32)
    balanced = np.asarray(balanced_section, dtype=np.float32)
    raw_clip = float(np.nanpercentile(np.abs(raw), 99.0))
    balanced_clip = float(np.nanpercentile(np.abs(balanced), 99.0))
    fig, axes = plt.subplots(1, 2, figsize=(15, 6), constrained_layout=True, sharey=True)
    extent = [float(horizontal_axis[0]), float(horizontal_axis[-1]), float(vertical_axis[-1]), float(vertical_axis[0])]
    axes[0].imshow(raw.T, aspect="auto", cmap="seismic", vmin=-raw_clip, vmax=raw_clip, extent=extent)
    axes[1].imshow(
        balanced.T,
        aspect="auto",
        cmap="seismic",
        vmin=-balanced_clip,
        vmax=balanced_clip,
        extent=extent,
    )
    axes[0].set_title(f"Raw | P99={raw_clip:.4g}")
    axes[1].set_title(f"Dynamic-gain balanced | P99={balanced_clip:.4g}")
    axes[0].set_ylabel("TVDSS (m)")
    for axis in axes:
        axis.set_xlabel("Line coordinate")
    fig.suptitle(title)
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.show()
    plt.close(fig)


In [ ]:
metrics = pd.read_csv(BATCH_METRICS_FILE)
selected = metrics.loc[metrics["well_name"].isin(SELECTED_WELLS)].copy()
if set(selected["well_name"]) != set(SELECTED_WELLS):
    missing = sorted(set(SELECTED_WELLS) - set(selected["well_name"]))
    raise ValueError(f"Selected wells are missing from batch metrics: {missing}")
if not selected["status"].eq("ok").all():
    raise ValueError("Every selected well must have status=ok.")
if (selected["corr"].astype(float) < MIN_LOCAL_CORR).any():
    raise ValueError("A selected well does not satisfy the minimum whole-well correlation.")

window_frames = [
    build_well_windows(row, CALIBRATION_WINDOW_M, CALIBRATION_STEP_M)
    for _, row in selected.iterrows()
]
training_windows = pd.concat(window_frames, ignore_index=True)
if len(training_windows) < 6:
    raise ValueError(f"Need at least six accepted calibration windows, got {len(training_windows)}.")

fit = fit_robust_log_gain(training_windows)
if not np.isfinite(fit["slope"]) or fit["slope"] <= 0.0:
    raise ValueError(f"Expected positive forward-gain/RMS slope, got {fit['slope']!r}.")

training_windows["log_relative_rms"] = np.log(training_windows["relative_rms"])
training_windows["log_forward_gain"] = np.log(training_windows["forward_gain"])
training_windows["predicted_forward_gain"] = np.exp(
    fit["intercept"] + fit["slope"] * training_windows["log_relative_rms"]
)
training_windows["balance_gain"] = np.power(
    training_windows["relative_rms"].to_numpy(dtype=np.float64),
    -fit["slope"],
)
training_windows.to_csv(TRAINING_SAMPLES_FILE, index=False)

q_low, q_high = np.percentile(
    training_windows["balance_gain"].to_numpy(dtype=np.float64),
    TRAINING_GAIN_QUANTILES,
)
balance_gain_clip = (
    max(float(ABSOLUTE_BALANCE_GAIN_CAP[0]), min(1.0, float(q_low))),
    min(float(ABSOLUTE_BALANCE_GAIN_CAP[1]), max(1.0, float(q_high))),
)
rms_floor = max(
    np.finfo(np.float32).tiny,
    0.5 * float(np.percentile(training_windows["relative_rms"], 2.0)),
)
reference_forward_gain = float(np.exp(fit["intercept"]))

survey = open_survey(SEISMIC_FILE, seismic_type="segy", segy_options=SEGY_OPTIONS)
geometry = survey.describe_geometry(domain="depth")
if geometry["sample_domain"] != "depth" or geometry["sample_unit"] != "m":
    raise ValueError("Expected depth-domain seismic with metre sample axis.")
sample_step_m = float(geometry["sample_step"])
application_window_samples = resolve_odd_window_samples(APPLICATION_WINDOW_M, sample_step_m)

fit_row = {
    **fit,
    "selected_wells": ",".join(SELECTED_WELLS),
    "n_training_windows": int(len(training_windows)),
    "calibration_window_m": CALIBRATION_WINDOW_M,
    "calibration_step_m": CALIBRATION_STEP_M,
    "application_window_m": APPLICATION_WINDOW_M,
    "application_window_samples": int(application_window_samples),
    "minimum_local_corr": MIN_LOCAL_CORR,
    "reference_forward_gain_at_relative_rms_1": reference_forward_gain,
    "balance_gain_clip_low": balance_gain_clip[0],
    "balance_gain_clip_high": balance_gain_clip[1],
    "relative_rms_floor": rms_floor,
    "balanced_transform": "raw_mean + (raw - raw_mean) * clip(relative_local_rms ** -slope)",
}
pd.DataFrame([fit_row]).to_csv(FIT_METRICS_FILE, index=False)
MODEL_FILE.write_text(json.dumps(fit_row, ensure_ascii=False, indent=2), encoding="utf-8")

print(pd.DataFrame([fit_row]).T)
print()
print(training_windows[[
    "well_name",
    "center_tvdss_m",
    "local_corr",
    "relative_rms",
    "forward_gain",
    "predicted_forward_gain",
    "balance_gain",
]])


In [ ]:
x = training_windows["log_relative_rms"].to_numpy(dtype=np.float64)
y = training_windows["log_forward_gain"].to_numpy(dtype=np.float64)
x_line = np.linspace(float(np.min(x)), float(np.max(x)), 200)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), constrained_layout=True)
scatter = axes[0].scatter(
    x,
    y,
    c=training_windows["local_corr"],
    cmap="viridis",
    vmin=0.0,
    vmax=1.0,
    s=42,
)
axes[0].plot(x_line, fit["intercept"] + fit["slope"] * x_line, color="black", lw=1.5)
axes[0].set_xlabel("ln(relative local RMS)")
axes[0].set_ylabel("ln(forward gain)")
axes[0].set_title(f"NW11 robust fit | slope={fit['slope']:.3f}")
axes[0].grid(True, alpha=0.25)
fig.colorbar(scatter, ax=axes[0], label="local correlation")

axes[1].plot(
    training_windows["center_tvdss_m"],
    training_windows["forward_gain"],
    marker="o",
    label="window target",
)
axes[1].plot(
    training_windows["center_tvdss_m"],
    training_windows["predicted_forward_gain"],
    marker="s",
    label="robust fit",
)
axes[1].set_xlabel("TVDSS (m)")
axes[1].set_ylabel("Forward gain")
axes[1].set_title("Forward gain along NW11")
axes[1].grid(True, alpha=0.25)
axes[1].legend()

axes[2].plot(
    training_windows["center_tvdss_m"],
    training_windows["balance_gain"],
    marker="o",
    color="tab:red",
)
axes[2].axhline(1.0, color="black", ls="--", lw=1.0)
axes[2].axhspan(balance_gain_clip[0], balance_gain_clip[1], color="tab:orange", alpha=0.15)
axes[2].set_xlabel("TVDSS (m)")
axes[2].set_ylabel("Relative inverse gain")
axes[2].set_title("Balanced-seismic correction")
axes[2].grid(True, alpha=0.25)

fit_figure = FIGURE_DIR / "qc_01_nw11_dynamic_gain_fit.png"
fig.savefig(fit_figure, dpi=180, bbox_inches="tight")
plt.show()
plt.close(fig)
print("Saved", fit_figure)


In [ ]:
expected_shape = (
    int(geometry["n_il"]),
    int(geometry["n_xl"]),
    int(geometry["n_sample"]),
)
is_smoke = SMOKE_TRACE_LIMIT > 0
trace_count = int(expected_shape[0] * expected_shape[1])
process_trace_count = min(trace_count, SMOKE_TRACE_LIMIT) if is_smoke else trace_count
if not is_smoke:
    for stale_smoke_path in (SMOKE_NPZ_FILE, FIGURE_DIR / "qc_02_smoke_raw_balanced.png"):
        if stale_smoke_path.exists():
            stale_smoke_path.unlink()

if is_smoke:
    print(f"Loading {process_trace_count} smoke traces...")
    smoke_indices = [
        (flat_index // expected_shape[1], flat_index % expected_shape[1])
        for flat_index in range(process_trace_count)
    ]
    smoke_traces = survey.read_traces_at_indices(smoke_indices, domain="depth")
    flat_raw = np.stack(
        [np.asarray(smoke_traces[index].values, dtype=np.float32) for index in smoke_indices],
        axis=0,
    )
    seismic_volume = None
else:
    print("Loading full seismic volume...")
    seismic_volume = import_seismic(
        SEISMIC_FILE,
        seismic_type="segy",
        iline=SEGY_OPTIONS["iline"],
        xline=SEGY_OPTIONS["xline"],
        istep=SEGY_OPTIONS["istep"],
        xstep=SEGY_OPTIONS["xstep"],
    )
    if tuple(seismic_volume.shape) != expected_shape:
        raise ValueError(f"Volume shape {seismic_volume.shape} != geometry shape {expected_shape}.")
    flat_raw = seismic_volume.reshape(-1, seismic_volume.shape[-1])

if is_smoke:
    balanced_storage = np.empty((process_trace_count, flat_raw.shape[1]), dtype=np.float32)
else:
    if TEMP_BALANCED_FILE.exists():
        TEMP_BALANCED_FILE.unlink()
    balanced_storage = np.memmap(
        TEMP_BALANCED_FILE,
        mode="w+",
        dtype=np.float32,
        shape=seismic_volume.shape,
    ).reshape(-1, flat_raw.shape[1])

sampled_gains: list[np.ndarray] = []
dead_trace_count = 0
raw_energy = 0.0
balanced_energy = 0.0
sample_count = 0

for start in range(0, process_trace_count, VOLUME_BATCH_TRACES):
    end = min(start + VOLUME_BATCH_TRACES, process_trace_count)
    raw_batch = flat_raw[start:end]
    balanced_batch, gain_batch, dead_count = apply_balance_to_batch(
        raw_batch,
        window_samples=application_window_samples,
        slope=fit["slope"],
        rms_floor=rms_floor,
        balance_gain_clip=balance_gain_clip,
    )
    balanced_storage[start:end] = balanced_batch
    dead_trace_count += dead_count
    raw_energy += float(np.sum(raw_batch.astype(np.float64) ** 2))
    balanced_energy += float(np.sum(balanced_batch.astype(np.float64) ** 2))
    sample_count += int(raw_batch.size)
    stride = max(1, gain_batch.size // 20000)
    sampled_gains.append(gain_batch.ravel()[::stride].astype(np.float32))
    if start == 0 or end == process_trace_count or (start // VOLUME_BATCH_TRACES) % 20 == 0:
        print(f"Processed traces {end}/{process_trace_count}")

gain_sample = np.concatenate(sampled_gains)
raw_rms = float(np.sqrt(raw_energy / max(sample_count, 1)))
balanced_rms = float(np.sqrt(balanced_energy / max(sample_count, 1)))

if is_smoke:
    np.savez_compressed(
        SMOKE_NPZ_FILE,
        raw=flat_raw[:process_trace_count].astype(np.float32),
        balanced=np.asarray(balanced_storage, dtype=np.float32),
        gain_sample=gain_sample,
    )
    print("Saved smoke artifact:", SMOKE_NPZ_FILE)
else:
    balanced_storage.flush()

print("Gain sample P1/P50/P99:", np.percentile(gain_sample, [1.0, 50.0, 99.0]))
print("Raw RMS:", raw_rms)
print("Balanced RMS:", balanced_rms)
print("Dead traces:", dead_trace_count)


In [ ]:
samples = np.asarray(geometry["sample_min"] + geometry["sample_step"] * np.arange(geometry["n_sample"]), dtype=np.float64)
ilines = np.asarray(geometry["inline_min"] + geometry["inline_step"] * np.arange(geometry["n_il"]), dtype=np.float64)
xlines = np.asarray(geometry["xline_min"] + geometry["xline_step"] * np.arange(geometry["n_xl"]), dtype=np.float64)

if is_smoke:
    raw_preview = flat_raw[:process_trace_count]
    balanced_preview = np.asarray(balanced_storage)
    count = min(64, process_trace_count)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True, sharey=True)
    raw_clip = float(np.percentile(np.abs(raw_preview[:count]), 99.0))
    balanced_clip = float(np.percentile(np.abs(balanced_preview[:count]), 99.0))
    axes[0].imshow(raw_preview[:count].T, aspect="auto", cmap="seismic", vmin=-raw_clip, vmax=raw_clip)
    axes[1].imshow(
        balanced_preview[:count].T,
        aspect="auto",
        cmap="seismic",
        vmin=-balanced_clip,
        vmax=balanced_clip,
    )
    axes[0].set_title("Raw smoke traces")
    axes[1].set_title("Dynamic-gain balanced smoke traces")
    axes[0].set_ylabel("Depth sample")
    smoke_figure = FIGURE_DIR / "qc_02_smoke_raw_balanced.png"
    fig.savefig(smoke_figure, dpi=180, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("Saved", smoke_figure)
else:
    balanced_volume = balanced_storage.reshape(seismic_volume.shape)
    nw11_row = selected.loc[selected["well_name"].eq("NW11")].iloc[0]
    nw11_inline_index = int(
        round((float(nw11_row["inline_float"]) - float(geometry["inline_min"])) / float(geometry["inline_step"]))
    )
    nw11_inline_index = int(np.clip(nw11_inline_index, 0, seismic_volume.shape[0] - 1))
    middle_xline_index = seismic_volume.shape[1] // 2

    plot_section_pair(
        seismic_volume[nw11_inline_index],
        balanced_volume[nw11_inline_index],
        horizontal_axis=xlines,
        vertical_axis=samples,
        title=f"NW11-nearest inline {ilines[nw11_inline_index]:g}",
        output_path=FIGURE_DIR / "qc_02_nw11_inline_raw_balanced.png",
    )
    plot_section_pair(
        seismic_volume[:, middle_xline_index, :],
        balanced_volume[:, middle_xline_index, :],
        horizontal_axis=ilines,
        vertical_axis=samples,
        title=f"Middle xline {xlines[middle_xline_index]:g}",
        output_path=FIGURE_DIR / "qc_03_middle_xline_raw_balanced.png",
    )


In [ ]:
segy_probe_max_abs = None
if not is_smoke:
    balanced_volume = balanced_storage.reshape(seismic_volume.shape)
    probe_flat_indices = [0, trace_count // 2, trace_count - 1]
    expected_probes = {
        index: np.asarray(balanced_volume.reshape(-1, balanced_volume.shape[-1])[index], dtype=np.float32).copy()
        for index in probe_flat_indices
    }

    if BALANCED_SEGY_FILE.exists():
        BALANCED_SEGY_FILE.unlink()
    textual = build_segy_textual_header(
        "NW11 calibrated dynamic-gain balanced depth seismic",
        [
            "input=mero_84_coord_extend",
            "domain=depth unit=m basis=TVDSS",
            "selected_wells=NW11",
            f"window_m={APPLICATION_WINDOW_M:.6g}",
            f"balance_exponent={fit['slope']:.6g}",
            f"balance_gain_clip={balance_gain_clip[0]:.6g},{balance_gain_clip[1]:.6g}",
            "transform=mean+(raw-mean)*clip(relative_local_rms**-exponent)",
            "xline_step=4 used for geometry addressing",
        ],
    )
    print("Writing SEG-Y:", BALANCED_SEGY_FILE)
    cigsegy.create_by_sharing_header(
        str(BALANCED_SEGY_FILE),
        str(SEISMIC_FILE),
        balanced_volume,
        keylocs=KEYLOCS,
        textual=textual,
    )

    reader = cigsegy.Pysegy(str(BALANCED_SEGY_FILE))
    probe_errors = []
    try:
        for index in probe_flat_indices:
            actual = np.asarray(
                reader.collect(index, index + 1, 0, balanced_volume.shape[-1]).squeeze(),
                dtype=np.float32,
            )
            probe_errors.append(float(np.max(np.abs(actual - expected_probes[index]))))
    finally:
        reader.close()
    segy_probe_max_abs = float(max(probe_errors))
    print("SEG-Y probe max abs error:", segy_probe_max_abs)

    balanced_storage.flush()
    del balanced_volume
    del balanced_storage
    if TEMP_BALANCED_FILE.exists():
        TEMP_BALANCED_FILE.unlink()

summary = {
    "status": "smoke_complete" if is_smoke else "complete",
    "sample_domain": "depth",
    "sample_unit": "m",
    "depth_basis": "tvdss",
    "selected_wells": list(SELECTED_WELLS),
    "source_seismic_file": str(SEISMIC_FILE.relative_to(repo_root)),
    "batch_metrics_file": str(BATCH_METRICS_FILE.relative_to(repo_root)),
    "calibration_window_m": CALIBRATION_WINDOW_M,
    "application_window_m": APPLICATION_WINDOW_M,
    "n_training_windows": int(len(training_windows)),
    "fit": fit,
    "reference_forward_gain": reference_forward_gain,
    "balance_gain_clip": [float(balance_gain_clip[0]), float(balance_gain_clip[1])],
    "gain_sample_percentiles": {
        "p01": float(np.percentile(gain_sample, 1.0)),
        "p50": float(np.percentile(gain_sample, 50.0)),
        "p99": float(np.percentile(gain_sample, 99.0)),
    },
    "raw_rms": raw_rms,
    "balanced_rms": balanced_rms,
    "dead_trace_count": int(dead_trace_count),
    "processed_trace_count": int(process_trace_count),
    "total_trace_count": int(trace_count),
    "balanced_segy_file": None if is_smoke else str(BALANCED_SEGY_FILE.relative_to(repo_root)),
    "segy_probe_max_abs_error": segy_probe_max_abs,
    "notes": [
        "The well-derived quantity is a forward gain; balanced seismic uses its relative inverse.",
        "Trace standardization is used only to estimate relative local RMS; output remains in source SEG-Y amplitude units.",
        "The balanced SEG-Y contains dynamic gain only; coherence is not multiplied into amplitudes.",
    ],
}
SUMMARY_FILE.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(summary, ensure_ascii=False, indent=2))
